# Amazon Bedrock APIs: Control Plane, Data Plane, and The "Versus" Breakdown

## 1. Theory & Real-World Context

To build production-grade AI applications on AWS, you must understand how Amazon Bedrock separates its infrastructure into two distinct API planes, and how to choose the right inference method for your workload.

### 1. Control Plane vs. Data Plane

* **Control Plane (`boto3.client('bedrock')`):** Used for configuration and management. You use this to list available models (`ListFoundationModels`), manage Provisioned Throughput, set up Guardrails, and trigger Programmatic Model Evaluation jobs.
* **Data Plane (`boto3.client('bedrock-runtime')`):** Used for actual AI inference. You use this to send prompts and receive completions or embeddings.

### 2. The Big "Versus": Which Inference API Should You Use?

| API Method | Payload Format | Execution Style | Real-World AI Engineering Use Case |
| --- | --- | --- | --- |
| **`InvokeModel`** | Model-Specific JSON (Bespoke) | Synchronous (Waits for full response) | Running batch embedding generation, image generation (SDXL/Titan), or automated data extraction workflows where the user isn't waiting on a UI. |
| **`InvokeModelWithResponseStream`** | Model-Specific JSON (Bespoke) | Streaming (Yields data in chunks) | Legacy chat applications where you need fast **Time to First Token (TTFT)**. The UI updates word-by-word, preventing timeouts on the frontend. |
| **`Converse` / `ConverseStream**` | **Standardized** (`messages`, `system`) | Synchronous OR Streaming | **The modern gold standard for text/multimodal LLMs.** Used for chatbots, virtual agents, and tool-use (Function Calling). You write the code once, and can swap Claude for Amazon Nova without changing your payload structure. |

### 3. Production Resiliency: Cross-Region Inference (CRIS)

Instead of hardcoding a region-specific model ID (e.g., `us-east-1`'s Claude model), you use a **Cross-Region Inference Profile ARN**. CRIS intelligently routes your prompt to multiple AWS regions (e.g., `us-east-1`, `us-west-2`, `us-east-2`) based on real-time availability.

* **Why it matters:** It prevents `HTTP 429 Too Many Requests` (Throttling) during peak traffic hours, doubling or tripling your effective throughput capacity.

---

## 2. Visual Architecture

```mermaid
flowchart TD
    subgraph App["AI Engineering Application (Python / Lambda)"]
        direction TB
        B1[boto3.client 'bedrock']
        B2[boto3.client 'bedrock-runtime']
    end

    subgraph ControlPlane["Control Plane (Management)"]
        LFM[ListFoundationModels API]
        EVAL[Model Evaluation Jobs]
    end

    subgraph DataPlane["Data Plane (Inference)"]
        direction LR
        INV[InvokeModel<br/>Raw JSON]
        CON[Converse / ConverseStream<br/>Standardized Messages]
    end

    subgraph CRIS["Cross-Region Inference (CRIS)"]
        R1[us-east-1]
        R2[us-west-2]
        R3[us-east-2]
    end

    B1 --> ControlPlane
    B2 --> DataPlane

    DataPlane -- "Inference Profile ARN" --> CRIS
    CRIS -- "Dynamic Routing" --> R1
    CRIS -- "Dynamic Routing" --> R2
    CRIS -- "Dynamic Routing" --> R3

```

---

## 3. Console Hands-On

**Objective:** Configure a Cross-Region Inference (CRIS) profile and find its ID for use in Boto3.

1. Open the **Amazon Bedrock Console**.
2. In the left navigation pane, scroll down to **Cross-region inference**.
3. You will see a list of **Inference Profiles** (e.g., `us.anthropic.claude-3-5-sonnet-20241022-v2:0`). Notice the `us.` prefix—this indicates it routes across all supported US regions.
4. Click on the profile name.
5. In the details page, look at the **System defined profile ARN**. This ARN (or simply the `us.` profile ID) is what you will pass into your `boto3` code instead of the standard model ID.
6. *Optional:* To view Model Evaluation, click **Model evaluation** in the left menu $\rightarrow$ **Create evaluation**. You can see how AWS allows you to benchmark models against datasets like toxicity or accuracy.

---


---

## 5. Practice Challenges

### Challenge 1: The Multi-Turn Chatbot (ConverseStream)

**Scenario:** A front-end application requires a chatbot that streams text to the UI as it generates, and retains conversation history.

* **Task:** Write a Python function using `client.converse_stream()`. Pass a `messages` array that contains one prior user prompt and assistant response, plus a new user prompt. Loop through the stream and capture the `contentBlockDelta` events to print the text chunks to the console.

### Challenge 2: Programmatic Model Evaluation Script

**Scenario:** Your MLOps team needs to verify that the latest Meta Llama model performs well on your proprietary summarization dataset before swapping it in production.

* **Task:** Using the Control Plane `bedrock` client, write a script that calls `create_evaluation_job()`. Configure it to use automatic evaluation (no human workers), target a specific S3 bucket for the input dataset, and measure the `ROBUSTNESS` and `ACCURACY` metrics.

---

## 6. Industry Standards & Tips

* **Always Use CRIS in Production:** Never use standard single-region `modelId` strings in production code (e.g., `anthropic.claude-3-sonnet...`). Always use the system-defined CRIS Profile ID (e.g., `us.anthropic.claude...` or `eu.amazon.nova...`). This is the easiest, zero-code way to implement high availability and bypass regional throttling.
* **UI / Frontend Integration:** If an AI response takes longer than 2 seconds, users perceive it as broken. Always use `ConverseStream` or `InvokeModelWithResponseStream` for anything user-facing. Time-To-First-Token (TTFT) is the most critical metric for AI UX.
* **Client Separation in Lambda:** A common AWS developer mistake is trying to call `invoke_model` on `boto3.client('bedrock')` and getting an `AttributeError`. Remember: `bedrock` is for administrative tasks; `bedrock-runtime` is for data (inference).
* **Model Evaluation for CI/CD:** Do not evaluate models purely by testing them in the console playground. For enterprise apps, integrate the `CreateEvaluationJob` API into your CI/CD pipeline (using Step Functions or GitHub Actions) to mathematically benchmark models against a golden dataset stored in S3 before promoting a new model version.

---

## 7. Interview Q&A (Targeted for AI Engineers)

### Q1: In your Python application, you need to list all available Bedrock models and then generate text. How do you initialize your Boto3 clients?

> **Answer:** You must initialize two separate clients. You use the control plane client `boto3.client('bedrock')` to call `list_foundation_models()`. For text generation, you must initialize the data plane client `boto3.client('bedrock-runtime')` and call `converse()` or `invoke_model()`.

### Q2: What is the architectural advantage of using the `Converse` API versus `InvokeModel`?

> **Answer:** `InvokeModel` requires constructing a bespoke JSON payload specific to the model provider (e.g., Titan's schema is different from Claude's). The `Converse` API provides a unified, standardized interface using a `messages` array. This abstracts the provider-specific differences, allowing developers to switch underlying models by simply changing the `modelId` parameter, drastically reducing technical debt.

### Q3: Your production Bedrock application starts receiving `HTTP 429 Too Many Requests` errors during peak business hours. How do you mitigate this without requesting a quota increase?

> **Answer:** Switch the application to use **Cross-Region Inference (CRIS)**. By replacing the standard model ID with a Cross-Region Inference Profile ARN (e.g., using `us.anthropic...`), AWS will automatically route the inference request to another supported region (like `us-west-2` or `us-east-2`) that currently has available capacity, vastly improving throughput and resiliency.

### Q4: When would you use `InvokeModelWithResponseStream` instead of the standard synchronous `InvokeModel`?

> **Answer:** Streaming is essential for user-facing applications (like chat interfaces) where generation might take several seconds. Instead of the user waiting in silence for the entire payload to finish, streaming yields data chunks in real-time, significantly improving the perceived latency and the Time-To-First-Token (TTFT). Synchronous invocation is better suited for asynchronous backend tasks, like batch processing or embedding generation.

## 4. Boto3 Implementation

This production-ready script demonstrates the separation of clients, streaming chunks, and using the modern `Converse` API.

```python
import json
import logging
from typing import List, Dict, Any
import boto3
from botocore.exceptions import ClientError

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger("bedrock_api_manager")

class BedrockAPIManager:
    """Demonstrates Control Plane vs Data Plane and standard vs streaming APIs."""

    def __init__(self, region_name: str = "us-east-1"):
        # Control Plane Client (Management)
        self.bedrock_admin = boto3.client("bedrock", region_name=region_name)
        
        # Data Plane Client (Inference)
        self.bedrock_runtime = boto3.client("bedrock-runtime", region_name=region_name)

    def list_active_text_models(self) -> List[str]:
        """Control Plane: Lists foundation models that support text generation."""
        try:
            logger.info("Fetching available foundation models...")
            response = self.bedrock_admin.list_foundation_models(
                byOutputModality="TEXT"
            )
            return [model["modelId"] for model in response.get("modelSummaries", [])]
        except ClientError as e:
            logger.error(f"Failed to list models: {e.response['Error']['Message']}")
            return []

    def run_streamed_inference(self, prompt: str, model_id: str) -> None:
        """
        Data Plane (Legacy/Raw): Uses InvokeModelWithResponseStream.
        Best for legacy models or bespoke payloads.
        """
        logger.info(f"Starting raw stream for {model_id}...")
        
        # Bespoke Titan payload
        payload = {
            "inputText": prompt,
            "textGenerationConfig": {"maxTokenCount": 512, "temperature": 0.5}
        }

        try:
            response = self.bedrock_runtime.invoke_model_with_response_stream(
                modelId=model_id,
                body=json.dumps(payload),
                accept="application/json",
                contentType="application/json"
            )

            stream = response.get("body")
            if stream:
                print("\n[Streaming Output]: ", end="", flush=True)
                for event in stream:
                    chunk = event.get("chunk")
                    if chunk:
                        # Decode the byte stream payload
                        decoded_chunk = json.loads(chunk.get("bytes").decode())
                        print(decoded_chunk.get("outputText", ""), end="", flush=True)
                print("\n")
                
        except ClientError as e:
            logger.error(f"Streaming failed: {e.response['Error']['Message']}")

    def converse_with_cris(self, prompt: str, cris_profile_id: str) -> None:
        """
        Data Plane (Modern): Uses Converse API with a Cross-Region Inference profile.
        """
        logger.info(f"Invoking Converse API using CRIS Profile: {cris_profile_id}")
        
        messages = [{"role": "user", "content": [{"text": prompt}]}]
        
        try:
            response = self.bedrock_runtime.converse(
                modelId=cris_profile_id, # Passes the 'us.' profile ID instead of a single region ID
                messages=messages,
                inferenceConfig={"temperature": 0.2, "maxTokens": 512}
            )
            
            output = response['output']['message']["content"][0]["text"]
            logger.info(f"Converse Output:\n{output}")
            
        except ClientError as e:
            logger.error(f"Converse API failed: {e.response['Error']['Message']}")

if __name__ == "__main__":
    manager = BedrockAPIManager()
    
    # 1. Control Plane check
    models = manager.list_active_text_models()
    logger.info(f"Found {len(models)} text models.")

    # 2. Raw Stream (Titan)
    manager.run_streamed_inference(
        prompt="Write a 3-sentence summary of cloud computing.",
        model_id="amazon.titan-text-express-v1"
    )

    # 3. Modern Converse with Cross-Region Inference
    # Using the 'us.' prefix indicates a CRIS Profile routing across multiple US regions.
    manager.converse_with_cris(
        prompt="Explain Cross-Region Inference in one sentence.",
        cris_profile_id="us.anthropic.claude-3-5-sonnet-20241022-v2:0"
    )

```
